<a href="https://colab.research.google.com/github/vinudi20251178-ux/Telco_churn_ml_cw/blob/main/notebooks/02_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Task 02

###part 02)

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
path = '/content/drive/MyDrive/Telco-Customer-Churn.csv'
df = pd.read_csv(path)


In [23]:
model_df = df.copy()
print("Original shape:", model_df.shape)


Original shape: (7043, 21)


In [25]:
model_df['TotalCharges'] = pd.to_numeric(model_df['TotalCharges'], errors='coerce')
model_df['TotalCharges'] = model_df['TotalCharges'].fillna(0)

In [26]:
model_df = model_df.drop(columns=['customerID'])

print("Updated shape:", model_df.shape)
print("Missing values:", model_df.isnull().sum().sum())
model_df.head()

Updated shape: (7043, 20)
Missing values: 0


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [27]:
from sklearn.model_selection import train_test_split

# Separate the input features (X) from the target (y)
X = model_df.drop(columns=['Churn'])
y = model_df['Churn']

# Convert the target to numbers: 1 = churned (positive class), 0 = stayed
y = y.map({'No': 0, 'Yes': 1})
print(y.value_counts())

# Split into training (80%) and test (20%) sets.
# stratify=y keeps the churn proportion the same in both sets; random_state makes the split reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Check the shapes and the churn proportion in each set
print("\nX_train:", X_train.shape, " X_test:", X_test.shape)
print("\nChurn proportion in training set:")
print(y_train.value_counts(normalize=True).round(3))
print("\nChurn proportion in test set:")
print(y_test.value_counts(normalize=True).round(3))

Churn
0    5174
1    1869
Name: count, dtype: int64

X_train: (5634, 19)  X_test: (1409, 19)

Churn proportion in training set:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64

Churn proportion in test set:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


The target variable Churn was converted to numbers (1 = churned, 0 = stayed), so that churn is the positive class when precision, recall and F1 are calculated later. The data was then split into a training set (80%, 5,634 customers) and a test set (20%, 1,409 customers).
The split is stratified on Churn, so both sets keep the original class balance of about 73.5% non-churners and 26.5% churners. This matters because the target is imbalanced, and an unstratified split could leave the sets with different proportions. A fixed random seed (42) makes the split reproducible. The test set is kept untouched until the final evaluation, and all preprocessing and tuning use the training set only.


In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Continuous features get scaled; text columns get one-hot encoded
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = X_train.select_dtypes(include='object').columns.tolist()

# drop='first' removes one redundant column per feature; handle_unknown='ignore' avoids errors on unseen categories
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'   # keeps SeniorCitizen (already 0/1) unchanged
)

# Fit on the training data only, then apply the same transformation to the test data (avoids data leakage)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print(preprocessor.get_feature_names_out())

Processed training shape: (5634, 30)
Processed test shape: (1409, 30)
['num__tenure' 'num__MonthlyCharges' 'num__TotalCharges'
 'cat__gender_Male' 'cat__Partner_Yes' 'cat__Dependents_Yes'
 'cat__PhoneService_Yes' 'cat__MultipleLines_No phone service'
 'cat__MultipleLines_Yes' 'cat__InternetService_Fiber optic'
 'cat__InternetService_No' 'cat__OnlineSecurity_No internet service'
 'cat__OnlineSecurity_Yes' 'cat__OnlineBackup_No internet service'
 'cat__OnlineBackup_Yes' 'cat__DeviceProtection_No internet service'
 'cat__DeviceProtection_Yes' 'cat__TechSupport_No internet service'
 'cat__TechSupport_Yes' 'cat__StreamingTV_No internet service'
 'cat__StreamingTV_Yes' 'cat__StreamingMovies_No internet service'
 'cat__StreamingMovies_Yes' 'cat__Contract_One year'
 'cat__Contract_Two year' 'cat__PaperlessBilling_Yes'
 'cat__PaymentMethod_Credit card (automatic)'
 'cat__PaymentMethod_Electronic check' 'cat__PaymentMethod_Mailed check'
 'remainder__SeniorCitizen']


Feature transformation
The three continuous features (tenure, MonthlyCharges, TotalCharges) were standardised, because they are on very different scales and KNN and Logistic Regression are sensitive to this. The 15 categorical features were one-hot encoded, dropping the first category of each to avoid redundant columns. SeniorCitizen was already 0/1, so it was passed through unchanged. This gives 30 features in total.
The preprocessor was fitted on the training set only and then applied to the test set, so no information from the test data influences the preprocessing (avoiding data leakage). The same preprocessed data is used for all three models. Random Forest does not need scaling, but it is not affected by it, so a single shared preprocessing step keeps the comparison fair.
